In [0]:
from pyspark.sql import functions as F
from silver_config import SILVER_COLUMNS

In [0]:
bazos_df = spark.read.table("bronze.bazos")

In [0]:
bazos_df.select("price").where(~F.col("price").rlike(r"\d+")).distinct().display()

In [0]:
bazos_df = bazos_df.withColumn("price", F.when(~F.col("price").rlike(r"\d+"), "n/a").otherwise(F.col("price")))

In [0]:
bazos_df = bazos_df.withColumn(
    "price_amount",
    F.when(
        F.col("price").rlike(r"\d"),
        F.regexp_replace(F.col("price"), r"[^0-9]", "").cast("int")
    ).otherwise(F.lit(None))
)

bazos_df = bazos_df.withColumn(
    "currency",
    F.when(
        F.col("price").rlike(r"\d"),
        F.trim(F.regexp_extract(F.col("price"), r"([^\d\s]+)$", 1))
    ).otherwise(F.lit("n/a"))
)

bazos_df = bazos_df.withColumn(
    "price_amount",
    F.when(
        F.col("price_amount") == 1,
        F.lit(None)
    ).otherwise(F.col("price_amount"))
)

bazos_df = bazos_df.drop("price")

In [0]:
bazos_df = bazos_df.withColumn("listing_id", F.col("listing_id").cast("int"))

In [0]:
bazos_df = bazos_df.withColumn(
    "posted_date", 
    F.to_date(
        F.regexp_replace(F.col("posted_date"), " ", ""),
        "d.M.yyyy"
    )
)

In [0]:
bazos_df = bazos_df.withColumn(
    "area",
    F.array_max(
        F.transform(
            F.regexp_extract_all(F.col("description"), F.lit(r"(\d+(?:,\d+)?)\s*m[²2]"), 1),
            lambda x: F.regexp_replace(x, ",", ".").cast("double")
        )
    )
)

bazos_df = bazos_df.withColumn(
    "rooms_count", F.regexp_replace(
        F.regexp_extract(F.col("description"), r"(\d+(?:,\d+)?)[\s-]?izbov", 1),
        ",",
        "."
    ).try_cast("float")
)

In [0]:
bazos_df = bazos_df.select(
    *SILVER_COLUMNS
)

In [0]:
bazos_df.write.mode("overwrite").saveAsTable("silver.bazos")